# OCI-hosted reranking with OCI Generative AI and VecDB search

Run a semantic search using Oracle VecDB, then rerank the returned candidates with a reranking model hosted by OCI Generative AI. The vector search stays in VecDB; the reranking request is sent to the OCI Generative AI RerankText API.

Run Sections 1 through 6 in order. Section 7 is optional cleanup.

## Prerequisites

- Install `oracle-vecdb`, `oci`, and `python-dotenv` in the notebook's Python environment.
- Copy `.env.example` to `.env` and fill in the VecDB connection and OCI configuration values.
- The OCI principal must be allowed to use Generative AI in the selected compartment.
- Use an OCI Generative AI region where the selected reranker is available on-demand. The default `cohere.rerank-v4.0-fast` uses `us-ashburn-1`; Phoenix currently exposes this model as dedicated-only. See the [regional availability table](https://docs.oracle.com/en-us/iaas/Content/generative-ai/model-endpoint-regions.htm).
- Have an embedding model available in VecDB. The default demo uses `ALL_MINILM_L12_V2`.

The default table is named `RERANK_SEARCH_DEMO`. It is safe to reuse for this notebook and can be removed in Section 7. This notebook does not require the in-database model-loading notebook.

## 1. Configure the demo

Load the shared `.env` file and define the OCI reranker, search table, query, and result count. Change only these optional settings when adapting the example to another integrated-embedding table.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "notebooks/vecdb/reranking/.env").is_file()),
    Path.cwd(),
)
env_file = repo / "notebooks/vecdb/reranking/.env"
if not env_file.is_file():
    raise FileNotFoundError("Copy .env.example to notebooks/vecdb/reranking/.env first.")
load_dotenv(env_file, override=True)

OCI_RERANK_MODEL = os.getenv("OCI_RERANK_MODEL", "").strip() or "cohere.rerank-v4.0-fast"
OCI_GENAI_REGION = os.getenv("OCI_GENAI_REGION", "").strip() or "us-ashburn-1"
SEARCH_TABLE = os.getenv("RERANK_SEARCH_TABLE", "RERANK_SEARCH_DEMO").strip()
SEARCH_EMBEDDING_MODEL = os.getenv("RERANK_SEARCH_EMBEDDING_MODEL", "ALL_MINILM_L12_V2").strip()
SEARCH_TEXT_FIELD = os.getenv("RERANK_SEARCH_TEXT_FIELD", "TEXT").strip()
SEED_DEMO_TABLE = os.getenv("RERANK_SEARCH_SEED", "true").lower() == "true"
QUERY = os.getenv("RERANK_SEARCH_QUERY", "How can Oracle Vector Database improve retrieval quality by reranking search results for RAG?")
TOP_K = int(os.getenv("RERANK_SEARCH_TOP_K", "6"))

required = ["VECDB_REST_URL", "OCI_CONFIG_FILE"]
missing = [name for name in required if not os.getenv(name)]
if not os.getenv("VECDB_ACCESS_TOKEN") and not (os.getenv("VECDB_USERNAME") and os.getenv("VECDB_PASSWORD")):
    missing.append("VECDB_ACCESS_TOKEN or VECDB_USERNAME/VECDB_PASSWORD")
if missing:
    raise ValueError("Set these values in .env: " + ", ".join(missing))

print(f"OCI reranker: {OCI_RERANK_MODEL}")
print(f"OCI Generative AI region: {OCI_GENAI_REGION}")
print(f"Search table: {SEARCH_TABLE}")
print(f"Query: {QUERY}")

## 2. Connect to VecDB and OCI Generative AI

Create a VecDB client for retrieval and an OCI Generative AI client for the external reranking call. A blank `OCI_COMPARTMENT_NAME` means the tenancy root compartment; otherwise the notebook resolves the exact child compartment name.

In [ ]:
import oci
from oracle_vecdb import Configuration, OracleVecDB
from oci.generative_ai_inference import GenerativeAiInferenceClient

connection = {"rest_url": os.environ["VECDB_REST_URL"]}
if os.getenv("VECDB_ACCESS_TOKEN"):
    connection["access_token"] = os.environ["VECDB_ACCESS_TOKEN"]
else:
    connection.update(username=os.environ["VECDB_USERNAME"], password=os.environ["VECDB_PASSWORD"])
vecdb_config = Configuration(**connection)
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    vecdb_config.verify_ssl = False
vecdb = OracleVecDB(vecdb_config)

oci_config = oci.config.from_file(
    file_location=os.path.expanduser(os.environ["OCI_CONFIG_FILE"]),
    profile_name=os.getenv("OCI_PROFILE", "DEFAULT"),
)
compartment_name = os.getenv("OCI_COMPARTMENT_NAME", "").strip()
if compartment_name:
    compartments = oci.identity.IdentityClient(oci_config).list_compartments(
        oci_config["tenancy"], compartment_id_in_subtree=True,
        access_level="ACCESSIBLE", lifecycle_state="ACTIVE",
    ).data
    matches = [item for item in compartments if item.name == compartment_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one accessible compartment named {compartment_name!r}; found {len(matches)}.")
    OCI_COMPARTMENT_ID = matches[0].id
else:
    OCI_COMPARTMENT_ID = oci_config["tenancy"]

genai_config = dict(oci_config)
genai_config["region"] = OCI_GENAI_REGION
genai = GenerativeAiInferenceClient(
    genai_config,
    service_endpoint=f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
print("VecDB and OCI Generative AI clients are ready.")

## 3. Prepare a searchable document set

The default path creates an integrated-embedding table and upserts six small documents. VecDB creates the document embeddings using the hosted embedding model. If `RERANK_SEARCH_SEED=false`, the table must already exist and use integrated embeddings; the notebook will query its existing records without changing them.

In [ ]:
documents = [
    {"id": "rerank-001", "metadata": {"TITLE": "Oracle Vector Database", "CATEGORY": "database", "TEXT": "Oracle Vector Database supports vector search, semantic retrieval, and reranking for retrieval-augmented generation applications."}},
    {"id": "rerank-002", "metadata": {"TITLE": "Two-stage retrieval", "CATEGORY": "search", "TEXT": "A vector search retrieves a broad candidate set, and a cross-encoder reranker then scores each query-document pair for relevance."}},
    {"id": "rerank-003", "metadata": {"TITLE": "Chocolate cake recipe", "CATEGORY": "recipe", "TEXT": "A chocolate cake recipe combines flour, eggs, cocoa, sugar, and butter before baking."}},
    {"id": "rerank-004", "metadata": {"TITLE": "Oracle database fundamentals", "CATEGORY": "database", "TEXT": "Oracle Database provides SQL, transactions, indexes, backup, and security features for enterprise workloads."}},
    {"id": "rerank-005", "metadata": {"TITLE": "RAG retrieval quality", "CATEGORY": "generative-ai", "TEXT": "RAG applications retrieve supporting passages first and use relevance scoring to select the evidence shown to a language model."}},
    {"id": "rerank-006", "metadata": {"TITLE": "OCI Object Storage", "CATEGORY": "oci", "TEXT": "OCI Object Storage stores private model artifacts and documents in buckets for applications and database services."}},
]

table_names = [item.table_name for item in (vecdb.list_vector_tables().items or [])]
if SEARCH_TABLE not in table_names:
    if not SEED_DEMO_TABLE:
        raise RuntimeError(f"Search table {SEARCH_TABLE!r} does not exist.")
    vecdb.create_vector_table(
        name=SEARCH_TABLE, comment="Small integrated-embedding corpus for OCI reranking demos",
        table_params={"auto_generate_id": False},
        embed_params={"model": SEARCH_EMBEDDING_MODEL, "embed_metadata_jsonpath": SEARCH_TEXT_FIELD},
        annotations={"TITLE": "string", "CATEGORY": "string", SEARCH_TEXT_FIELD: "string"},
    )
    print(f"Created integrated-embedding table: {SEARCH_TABLE}")
elif SEED_DEMO_TABLE:
    table = vecdb.describe_vector_table(name=SEARCH_TABLE)
    if not table.embed_params:
        raise RuntimeError(f"{SEARCH_TABLE} is not an integrated-embedding table. Set RERANK_SEARCH_SEED=false and adapt the search cell for BYOV vectors.")

if SEED_DEMO_TABLE:
    vecdb.upsert_vectors(table_name=SEARCH_TABLE, vectors=documents)
    print(f"Upserted {len(documents)} demo documents.")
else:
    print(f"Using existing table without seeding: {SEARCH_TABLE}")

## 4. Run the VecDB semantic search

Ask VecDB for the initial candidate set. This is the retrieval stage: the table's integrated embedding model turns the query into a vector and VecDB returns the nearest documents.

In [ ]:
def query_items(response):
    if isinstance(response, (list, tuple)):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []

def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)

def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})

def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance", item.get("score"))
    return getattr(item, "distance", getattr(item, "score", None))

search_response = vecdb.query(table_name=SEARCH_TABLE, query_by={"text": QUERY}, top_k=TOP_K, include_vectors=False)
search_items = query_items(search_response)
if not search_items:
    raise RuntimeError("VecDB returned no search results.")

print("Initial VecDB search results:")
for rank, item in enumerate(search_items, 1):
    meta = result_metadata(item)
    print(f"{rank}. distance={result_distance(item)}  {result_id(item)}  {meta.get(SEARCH_TEXT_FIELD, '')}")

## 5. Rerank the search results with OCI Generative AI

Pass the query and only the documents returned by VecDB to OCI's `RerankText` API. The service scores each query-document pair and returns the final relevance ordering. This is external reranking; no reranker is loaded into VecDB. See the [OCI RerankText API](https://docs.oracle.com/en-us/iaas/tools/python/latest/api/generative_ai_inference/client/oci.generative_ai_inference.GenerativeAiInferenceClient.html).

In [ ]:
candidate_documents = [result_metadata(item).get(SEARCH_TEXT_FIELD, "") for item in search_items]
if any(not text for text in candidate_documents):
    raise ValueError(f"Search results must contain text in metadata field {SEARCH_TEXT_FIELD!r}.")

request = oci.generative_ai_inference.models.RerankTextDetails(
    input=QUERY,
    compartment_id=OCI_COMPARTMENT_ID,
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=OCI_RERANK_MODEL),
    documents=candidate_documents,
    top_n=len(candidate_documents),
    is_echo=True,
)
try:
    rerank_response = genai.rerank_text(request)
except oci.exceptions.ServiceError as exc:
    if exc.status == 404:
        raise RuntimeError(f"OCI could not find {OCI_RERANK_MODEL!r} in {OCI_GENAI_REGION!r}. Choose an on-demand region in .env or use a dedicated endpoint.") from exc
    raise

rerank_items = rerank_response.data.document_ranks or []
if not rerank_items:
    raise RuntimeError("OCI Generative AI returned no document ranks.")

print(f"Reranked results from OCI Generative AI model {OCI_RERANK_MODEL}:")
reranked_ids = []
for rank, item in enumerate(rerank_items, 1):
    index = int(item.index)
    score = float(item.relevance_score)
    source = search_items[index]
    reranked_ids.append(result_id(source))
    print(f"{rank}. score={score:.6f}  {result_id(source)}  {candidate_documents[index]}")

print("Success: VecDB search candidates were reranked by OCI Generative AI.")
if [result_id(item) for item in search_items] == reranked_ids:
    print("The order did not change for this query; OCI still scored every candidate.")
else:
    print("OCI Generative AI changed the candidate order.")

## 6. What the two stages demonstrate

Vector search is the fast first-stage retrieval step: it produces a manageable candidate set. OCI Generative AI reranking is the precision step: the hosted cross-encoder evaluates the query against each candidate and returns the final order. In an application, use the reranked order when building the prompt or response.

## 7. Optional cleanup

Run this cell only if you used the default `RERANK_SEARCH_DEMO` table and want to remove the demo table. OCI Generative AI models are managed by Oracle and are not removed by this notebook.

In [ ]:
if SEARCH_TABLE == "RERANK_SEARCH_DEMO":
    table_names = [item.table_name for item in (vecdb.list_vector_tables().items or [])]
    if SEARCH_TABLE in table_names:
        vecdb.drop_vector_table(name=SEARCH_TABLE)
        print(f"Dropped demo table: {SEARCH_TABLE}")
    else:
        print(f"Demo table does not exist: {SEARCH_TABLE}")
else:
    print("Cleanup skipped because SEARCH_TABLE is not RERANK_SEARCH_DEMO.")